In [0]:
%run ../../00_common/data_utils

In [0]:
# Clean 子表配置：(表名, srcc_id 列名, mrkt_code 列名)
CLEAN_SUB_TABLES = [
    ("t_clean_address", "srca_srcc_id", "srca_mrkt_code"),
    ("t_clean_auxiliary_attribute", "sraa_srcc_id", "sraa_mrkt_code"),
    ("t_clean_consumergroup", "srcg_srcc_id", "srcg_mrkt_code"),
    ("t_clean_crossbrand_optin", "srbo_srcc_id", "srbo_mrkt_code"),
    ("t_clean_custom_attributes", "srat_srcc_id", "srat_mrkt_code"),
    ("t_clean_emedia", "srce_srcc_id", "srce_mrkt_code"),
    ("t_clean_hair_concerns", "srhc_srcc_id", "srhc_mrkt_code"),
    ("t_clean_hair_type", "srht_srcc_id", "srht_mrkt_code"),
    ("t_clean_hobby", "srhb_srcc_id", "srhb_mrkt_code"),
    ("t_clean_makeup_concerns", "srmc_srcc_id", "srmc_mrkt_code"),
    ("t_clean_notes", "srno_srcc_id", "srno_mrkt_code"),
    ("t_clean_optin", "srco_srcc_id", "srco_mrkt_code"),
    ("t_clean_phone", "srcp_srcc_id", "srcp_mrkt_code"),
    ("t_clean_program", "srpg_srcc_id", "srpg_mrkt_code"),
    ("t_clean_remark", "srcr_srcc_id", "srcr_mrkt_code"),
    ("t_clean_skin_concerns", "srsk_srcc_id", "srsk_mrkt_code"),
    ("t_clean_terms", "srct_srcc_id", "srct_mrkt_code"),
]

In [0]:
def load_anonymization_keys(process_date):
    """
    读取 t_mdm_anonymization_log，按 create_time 日期过滤，
    返回去重后的 business key。
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    secret_key = get_env_config('anonymization_aes_key')
    log_table = f"{anonymization_db}.t_mdm_anonymization_log"

    log_df = (
        spark.table(log_table)
        .where(F.to_date(F.col("create_time")) == F.lit(process_date))
        .where(F.col("status") == ANON_STATUS_COMPLETE)
        .select(
            "MarketCode",
            "BrandCode",
            "SourceSystemCode",
            F.aes_decrypt(F.unhex(F.col("ConsumerId")), F.lit(secret_key)).cast("string").alias("ConsumerId"),
        )
        .distinct()
    )

    log_df = log_df.checkpoint(eager=True)

    log_count = log_df.count()
    print(f"load_anonymization_keys: found {log_count} log rows for date={process_date}")

    if log_count == 0:
        print("load_anonymization_keys: no records for this date, return empty")
        return spark.createDataFrame(
            [],
            "MarketCode STRING, BrandCode STRING, SourceSystemCode STRING, ConsumerId STRING"
        )
    return log_df

In [0]:
def match_clean_keys_df(keys_df):
    """
    通过 business key semi join t_clean_consumer (Delta path) 获取 srcc_id。
    """
    cleansed_path = get_env_config('consumer_archive_path')
    clean_consumer_path = f"{cleansed_path}/t_clean_consumer"

    # 先用 business key 做 semi join 裁剪
    clean_business_keys = (
        keys_df
        .select(
            F.col("MarketCode").alias("SRCC_MRKT_CODE"),
            F.col("BrandCode").alias("SRCC_BRND_CODE"),
            F.col("SourceSystemCode").alias("SRCC_SRCS_CODE"),
            F.col("ConsumerId").alias("SRCC_CONSUMERID"),
        )
        .distinct()
    )

    clean_consumer_df = (
        spark.read.format("delta").load(clean_consumer_path)
        .join(
            F.broadcast(clean_business_keys),
            ["SRCC_MRKT_CODE", "SRCC_BRND_CODE", "SRCC_SRCS_CODE", "SRCC_CONSUMERID"],
            "semi"
        )
    )

    # semi join 已过滤出匹配的行，直接取列即可，无需二次 inner join
    return (
        clean_consumer_df
        .select(
            F.col("SRCC_ID").alias("srcc_id"),
            F.col("SRCC_MRKT_CODE").alias("srcc_mrkt_code"),
            F.col("SRCC_BRND_CODE").alias("srcc_brnd_code"),
            F.col("SRCC_SRCS_CODE").alias("srcc_srcs_code"),
            F.col("SRCC_CONSUMERID").alias("srcc_consumerid"),
        )
        .distinct()
    )

In [0]:
def delete_clean_tables(clean_keys_df):
    """
    删除 clean 层：17 张子表 + t_clean_consumer 主表（均通过 Delta path）。
    子表按 {fk_col} = srcc_id AND {mrkt_col} = srcc_mrkt_code 删除；
    主表按 4 列 business key 删除。
    顺序：先子表，后主表。
    """
    cleansed_path = get_env_config('consumer_archive_path')

    if clean_keys_df.isEmpty():
        print("delete_clean_tables: no clean consumer matches, skip")
        return

    clean_id_df = clean_keys_df.select(F.col("srcc_id"), F.col("srcc_mrkt_code")).distinct()

    # 删 clean 子表
    for table_name, fk_col, mrkt_col in CLEAN_SUB_TABLES:
        table_path = f"{cleansed_path}/{table_name}"
        try:
            (
                DeltaTable.forPath(spark, table_path).alias("target")
                .merge(
                    clean_id_df.alias("source"),
                    f"target.{fk_col} = source.srcc_id AND target.{mrkt_col} = source.srcc_mrkt_code"
                )
                .whenMatchedDelete()
                .execute()
            )
            print(f"delete_clean_tables: processed {table_path}")
        except Exception as e:
            raise RuntimeError(f"delete_clean_tables failed on {table_path}: {e}") from e

    # 最后删 clean 主表
    clean_consumer_path = f"{cleansed_path}/t_clean_consumer"
    try:
        (
            DeltaTable.forPath(spark, clean_consumer_path).alias("target")
            .merge(
                clean_keys_df.alias("source"),
                """
                target.SRCC_MRKT_CODE = source.srcc_mrkt_code AND
                target.SRCC_BRND_CODE = source.srcc_brnd_code AND
                target.SRCC_SRCS_CODE = source.srcc_srcs_code AND
                target.SRCC_CONSUMERID = source.srcc_consumerid
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        print(f"delete_clean_tables: processed {clean_consumer_path}")
    except Exception as e:
        raise RuntimeError(f"delete_clean_tables failed on {clean_consumer_path}: {e}") from e

In [0]:
def delete_clean_landing(clean_id_df):
    """
    删除 landing 层 consumerlist_raw 中对应 srcc_id 的记录。
    """
    if clean_id_df.isEmpty():
        print("delete_clean_landing: no clean consumer matches, skip")
        return

    cleansed_path = get_env_config('consumer_archive_path')
    landing_path = f"{cleansed_path}/consumerlist_raw"
    try:
        (
            DeltaTable.forPath(spark, landing_path).alias("target")
            .merge(
                clean_id_df.alias("source"),
                "target.slndc_id = source.srcc_id"
            )
            .whenMatchedDelete()
            .execute()
        )
        print(f"delete_clean_landing: {landing_path} processed")
    except Exception as e:
        raise RuntimeError(f"delete_clean_landing failed on {landing_path}: {e}") from e

In [0]:
def delete_clean_records_for_archive(process_date):
    """
    加载 keys → 解析 clean consumer ID → 删除 clean 层数据。
    """
    print(f"1. Loading anonymization keys for process_date={process_date}")
    keys_df = load_anonymization_keys(process_date)
    keys_df = keys_df.checkpoint(eager=True)

    if keys_df.isEmpty():
        print("No anonymization keys for this date, nothing to delete")
        return

    print("2.1 Resolving clean consumer matches")
    clean_keys_df = match_clean_keys_df(keys_df)
    clean_keys_df = clean_keys_df.checkpoint(eager=True)

    clean_count = clean_keys_df.count()
    print(f"2.2 Resolved {clean_count} clean consumers to delete")

    print("3.1 Deleting clean tables")
    delete_clean_tables(clean_keys_df)

    clean_id_df = clean_keys_df.select(F.col("srcc_id")).distinct()
    print("3.2 Deleting clean landing records")
    delete_clean_landing(clean_id_df)

    print("[completed]")

In [0]:
task_id = dbutils.widgets.get("task_id")
process_date = get_ex_param("process_date", (datetime.utcnow() - timedelta(days=1)).strftime("%Y-%m-%d"))

print(f"task_id: {task_id}")
print(f"process_date: {process_date}")

step_name = "delete_records_for_archive"
step_num = "03-archive"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    delete_clean_records_for_archive(process_date)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )